<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 04</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Made to order</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">Compose one model from an MLIP, electrostatics and dispersion, and see what each term adds.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> Meet your agent; Ready-made<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

Some research needs something beyond defaults. You will meet these classes
along the way:

- `AIMNet2Wrapper`: a built-in model you complete with its interaction terms
- `PipelineModelWrapper`: joins several models into one

<img src="assets/agent04-roadmap-light.svg" alt="Sections of this notebook on one line." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

Intermolecular interactions drive molecular recognition, solvation, crystal
packing and adsorption. Electrostatics is often their largest part: it is
strong, long-ranged, and sets how charged and polar molecules bind.
Dispersion is weaker and acts in every pair. Advanced potentials such as
AIMNet2 describe both better with explicit physical terms.

<img src="assets/agent04-top-light.svg" alt="A chef tosses three ingredients, a trained network, D3 dispersion and electrostatics, into a pot that an agent stirs; they come out as one composed model that takes a structure and returns energy and forces." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

In [ ]:
# Load the course helpers and print what your agent needs to start
from helpers import agent04 as helpers
helpers.configure_tutorial()             # course display defaults
work_dir = helpers.make_work_dir()       # notebooks/agent04_work
helpers.show_session()                   # prints the session lines

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Compose a model from interaction terms</h2>

A pipeline adds models together or passes one model's output to the next.
Here the MLIP, the machine-learned interatomic potential AIMNet2, predicts
a charge on every atom, the electrostatics term uses those charges, and D3
adds dispersion as a separate term. As in the kitchen above, each
ingredient adds what the others lack.

#### Example: AIMNet2 as it comes

```text
use the nvalchemi-toolkit skill
- show me how well AIMNet2 alone describes pairs of molecules
- model: AIMNet2, through toolkit's built-in wrapper, as it comes
- structures: helpers.load_nci_scans() gives a list of ASE atoms and scan_labels; each scan point has the pair AB and the molecules A and B
- keep the total charge of each structure
- first tell me which terms the built-in wrapper leaves out
- write agent04_work/compose_build.py
- give me aimnet_model: the loaded model
- give me scan_batch: all structures in one batch on the gpu
- give me scan_labels: unchanged
- give me base_energy: from aimnet_model, shape (n_structures, 1), in eV
```

In [ ]:
%load agent04_work/compose_build.py

In [ ]:
# Plot AIMNet2 as it comes against the reference
helpers.plot_interaction_curves(scan_labels, base=base_energy)

Each panel follows one pair as its separation changes; the points are the
state-of-the-art reference.

#### Example: add D3 and electrostatics

```text
- now add what AIMNet2 leaves out: D3 dispersion and electrostatics
- dispersion: toolkit's D3 wrapper; helpers.d3_settings() has the parameters that belong to this AIMNet2 checkpoint
- electrostatics: helpers.DirectCoulombAdapter, a finite coulomb term for isolated molecules; it must use the charges AIMNet2 predicts
- forces have to stay consistent with the summed energy
- first tell me how the charges get from AIMNet2 to the coulomb term
- write agent04_work/compose_run.py
- give me composed_model: one model with all three terms
- give me composed_energy: for scan_batch, shape (n_structures, 1), in eV
- give me ncia_energy: from composed_model, for the structures of helpers.load_ncia() in the loader's order, shape (n_ncia_structures, 1), in eV
- no plots, no loop over structures
```

In [ ]:
%load agent04_work/compose_run.py

In [ ]:
# Add the composed model to the same panels
helpers.plot_interaction_curves(scan_labels, base=base_energy, composed=composed_energy)

The composed curve lies on the reference points where the bare MLIP
does not. For the 205 pairs of the NCIA set, the scatter plot sets the same
model's interaction energy against the reference.

> </> API
>
> `DFTD3ModelWrapper`
>
> **Input:** the D3 parameters<br>
> **Effect:** dispersion as an independent term of the model

In [ ]:
# Compare the saved energies of the 205 pairs with their reference values
helpers.plot_ncia_accuracy(ncia_energy)

Each point is one pair; the title gives the mean absolute error.

> ⚙️ **Fine control**
>
> ```python
> PipelineModelWrapper(groups=[
>     PipelineGroup(                      # steps run in order
>         steps=[PipelineStep(aimnet, wire={"charges": "partial_charges"}),
>                coulomb],
>         use_autograd=True,              # forces from the summed energy
>     ),
>     PipelineGroup(steps=[d3], use_autograd=False),   # independent term
> ])
> ```
>
> Example: `build it again with a 12 Å cutoff for the coulomb term`

The two terms also act inside a single molecule. The table gives the largest
force on an atom of ethanol at ASE's built-in geometry, for the bare MLIP and
with each term added alone or together, and the O-H length after relaxing
with the bare MLIP and with both terms.

In [ ]:
# Compare the bare MLIP and the composed model on ethanol
helpers.show_single_molecule_check(aimnet_model, composed_model)

With the bare MLIP the largest force on an atom of ethanol is 4.44 eV/Å;
with both terms it is 0.78 eV/Å. Relaxed without its electrostatics term,
the molecule settles into the wrong shape.

Composing is what makes this model whole. The bare MLIP supplies the
short-range chemistry, electrostatics and dispersion supply what it was
trained to leave out, and Toolkit joins them into one model that every
relaxation and simulation can use.

#### Try it: compose a model without D3

What does each term contribute? One way to find out is to leave one out.
Ask your agent to compose AIMNet2 with electrostatics and no D3, and
evaluate the same structures.

| | |
| --- | --- |
| input  | `scan_batch`; `aimnet_model`; `helpers.DirectCoulombAdapter` |
| output | `no_d3_model`, `no_d3_energy`, in `agent04_work/no_d3.py` |

Done when: the plot adds your curve to the three panels, beside the complete model and the reference.

> 💡 **Hint**
>
> Nothing at hand? `helpers.load_nci_scans()` gives you a tested default.

In [ ]:
%load agent04_work/no_d3.py

In [ ]:
# Add your model without D3 to the same panels
helpers.plot_interaction_curves(scan_labels, base=base_energy, composed=composed_energy, no_d3=no_d3_energy)

Without D3 the error rises from 0.34 to 2.03 kcal/mol. D3 on its own
does not help either: the bare MLIP binds the polar pairs too strongly, and
D3 adds more attraction. Electrostatics corrects the overbinding and leaves
the pairs slightly underbound, and D3 supplies the attraction that is still
missing. The model needs both terms.

#### Try it: find the lowest conformer

The composed model relaxes a batch as a built-in one does.
One molecule can fold in many ways. Ask your agent to relax all the
conformers of alanine dipeptide at once with the model you composed, and
rank them by energy.

| | |
| --- | --- |
| input  | `helpers.load_dipeptide_grid()`; `composed_model`; `helpers.relax_settings()` |
| output | `conformer_batch`, `relaxed_conformers`, `conformer_energy`, in `agent04_work/conformers.py` |

Done when: the map shows where each conformer ended.

> 💡 **Hint**
>
> `helpers.show_conformers(...)` draws the result. `conformer_batch` is a batch from the loader; `relaxed_conformers` is a batch, in the loader's order; `conformer_energy` is one energy per conformer, in eV, in the same order.

> 💡 **Hint**
>
> Nothing at hand? `helpers.load_dipeptide_grid()` gives you a tested default.

In [ ]:
%load agent04_work/conformers.py

In [ ]:
# Map where each conformer started and ended
helpers.show_conformers(conformer_batch, relaxed_conformers, conformer_energy)

The viewer under the map shows the conformer with the lowest energy; the
map itself plots phi against psi.
Many starting points end in the same basin, so the map shows far fewer end
points than starts. The labelled markers are minima published for this
molecule in the gas phase (Mironov et al., J. Comput. Chem. 40, 297, 2019;
MP2/6-311++G** in vacuo). The marker labelled beta2 is a minimum in the published calculation but not for
this model: a structure started there rests on a flat shelf and, relaxed tightly, slides into a
neighbouring basin.

> </> API
>
> `FIRE2(model=..., dt=..., n_steps=..., convergence_hook=...)`
>
> **Input:** a model, a step budget and a stopping rule<br>
> **Effect:** relaxes every structure of a batch at once

> </> API
>
> `composed_model.make_neighbor_hooks()`
>
> **Input:** none; call it on the composed model<br>
> **Effect:** gives the hooks that build its neighbor lists; register them before the run

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

Toolkit joins models with a plus sign, and has long-range electrostatics of its own.

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Join models with <code>+</code></b></div><div style="margin: 2px 0 0;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/07_composable_model_composition.py">07_composable_model_composition.py</a></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Two independent models summed in one line.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>Ewald electrostatics</b></div><div style="margin: 2px 0 0;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/06_ewald_electrostatics.py">06_ewald_electrostatics.py</a></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Long-range electrostatics for a periodic ionic crystal.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b>AIMNet2 with Ewald</b></div><div style="margin: 2px 0 0;"><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/08_aimnet2_ewald_pipeline.py">08_aimnet2_ewald_pipeline.py</a></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">AIMNet2 charges feed Ewald in one pipeline, then a batched relaxation.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 04 recap</h2><div style="margin: 0 0 10px;">A model is made to order when you join the terms your chemistry needs. AIMNet2 with electrostatics and dispersion is one such model, and it serves every relaxation and simulation as a built-in one does. The next notebook brings in a model Toolkit has no wrapper for.</div><div style="margin: 0;"><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;"><b>Next:</b> <a href="alchemi-agent-05-guest-model.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Guest model →</a></div></div>